# Phase 2 — Feature Engineering (Kaggle GPU)
**Stream A:** FinBERT sentiment features | **Stream B:** Technical indicators (pure NumPy/pandas — no pandas-ta)  
Upload `market_data.db` as a Kaggle dataset named `market-data-db` before running.

In [ ]:
# ── 1. Imports & GPU check ────────────────────────────────────────────────────
# No extra installs needed — only stdlib + packages pre-installed on Kaggle.
import sqlite3, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import BertTokenizer, BertForSequenceClassification
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── 2. Paths ──────────────────────────────────────────────────────────────────
DB_PATH  = Path('/kaggle/input/market-data-db/market_data.db')
OUT_PATH = Path('/kaggle/working/feature_matrix.csv')

assert DB_PATH.exists(), (
    f'DB not found at {DB_PATH}. '
    'Did you attach the dataset? Notebook → Add Data → your dataset slug.'
)
print(f'DB found: {DB_PATH} ({DB_PATH.stat().st_size / 1e3:.1f} KB)')

In [ ]:
# ── 3. FinBERT loader ─────────────────────────────────────────────────────────
MODEL_NAME = 'ProsusAI/finbert'

print(f'[Stream A] Loading {MODEL_NAME} on {DEVICE} …')
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
finbert   = BertForSequenceClassification.from_pretrained(MODEL_NAME)
finbert.eval().to(DEVICE)

BATCH_SIZE = 64 if DEVICE == 'cuda' else 16
print(f'Batch size: {BATCH_SIZE}')

In [ ]:
# ── 4. FinBERT inference ──────────────────────────────────────────────────────
def score_texts(texts: list, batch_size: int = BATCH_SIZE) -> list:
    """Returns [{positive, negative, neutral}, …] for every input text."""
    results = []
    for i in tqdm(range(0, len(texts), batch_size), desc='FinBERT batches'):
        batch = texts[i : i + batch_size]
        enc   = tokenizer(batch, return_tensors='pt', padding=True,
                          truncation=True, max_length=512).to(DEVICE)
        with torch.no_grad():
            logits = finbert(**enc).logits
        probs = F.softmax(logits, dim=-1).cpu().numpy()
        # ProsusAI/finbert label order: positive=0, negative=1, neutral=2
        for row in probs:
            results.append({'positive': float(row[0]),
                            'negative': float(row[1]),
                            'neutral':  float(row[2])})
    return results

In [ ]:
# ── 5. Stream A — Sentiment features ─────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)

hl = pd.read_sql("""
    SELECT ticker,
           DATE(published_at) AS date,
           title || ' ' || COALESCE(description, '') AS text
    FROM headlines
    WHERE ticker IS NOT NULL AND title IS NOT NULL
""", conn)

rd = pd.read_sql("""
    SELECT ticker,
           DATE(datetime(CAST(created_utc AS REAL), 'unixepoch')) AS date,
           title || ' ' || COALESCE(body, '') AS text
    FROM reddit_posts
    WHERE ticker IS NOT NULL AND title IS NOT NULL
""", conn)
conn.close()

texts_df = pd.concat([hl, rd], ignore_index=True)
texts_df['text'] = texts_df['text'].str.strip().replace('', np.nan)
texts_df = texts_df.dropna(subset=['text']).reset_index(drop=True)
print(f'Texts to score: {len(texts_df)}')

In [ ]:
# Score + aggregate
scores_df = pd.DataFrame(score_texts(texts_df['text'].tolist()))
texts_df  = pd.concat([texts_df, scores_df], axis=1)
texts_df['net_sentiment'] = texts_df['positive'] - texts_df['negative']

sentiment_df = (
    texts_df
    .groupby(['ticker', 'date'])
    .agg(
        mean_sentiment      =('net_sentiment', 'mean'),
        sentiment_volatility=('net_sentiment', 'std'),
        headline_volume     =('net_sentiment', 'count'),
        mean_positive       =('positive',      'mean'),
        mean_negative       =('negative',      'mean'),
        mean_neutral        =('neutral',        'mean'),
    )
    .reset_index()
)
sentiment_df['sentiment_volatility'] = sentiment_df['sentiment_volatility'].fillna(0.0)
sentiment_df['negative_spike_flag']  = (sentiment_df['mean_negative'] > 0.5).astype(int)

print(f'[Stream A] Sentiment shape: {sentiment_df.shape}')
sentiment_df.head()

In [ ]:
# ── 6. Stream B helpers — pure NumPy/pandas implementations ──────────────────
# All functions operate on a single-ticker DataFrame sorted ascending by date.

def compute_rsi(close: pd.Series, length: int = 14) -> pd.Series:
    """Wilder-smoothed RSI (matches TA-Lib / pandas-ta output)."""
    delta = close.diff()
    gain  = delta.clip(lower=0)
    loss  = (-delta).clip(lower=0)
    # First average: simple mean over the first `length` bars
    avg_gain = gain.ewm(alpha=1 / length, min_periods=length, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / length, min_periods=length, adjust=False).mean()
    rs  = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi


def compute_macd(
    close: pd.Series, fast: int = 12, slow: int = 26, signal: int = 9
) -> pd.DataFrame:
    """Returns columns: MACD_12_26_9, MACDh_12_26_9, MACDs_12_26_9."""
    ema_fast   = close.ewm(span=fast,   adjust=False).mean()
    ema_slow   = close.ewm(span=slow,   adjust=False).mean()
    macd_line  = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram  = macd_line - signal_line
    return pd.DataFrame({
        'MACD_12_26_9':  macd_line,
        'MACDs_12_26_9': signal_line,
        'MACDh_12_26_9': histogram,
    })


def compute_bbands(
    close: pd.Series, length: int = 20, std: float = 2.0
) -> pd.DataFrame:
    """
    Returns columns matching pandas-ta naming:
    BBL_20_2.0, BBM_20_2.0, BBU_20_2.0, BBB_20_2.0 (bandwidth), BBP_20_2.0 (percent-b).
    """
    mid   = close.rolling(length).mean()
    sigma = close.rolling(length).std(ddof=0)   # population std, same as pandas-ta
    upper = mid + std * sigma
    lower = mid - std * sigma
    bwidth = (upper - lower) / mid.replace(0, np.nan)  # normalised bandwidth
    pctb   = (close - lower) / (upper - lower).replace(0, np.nan)
    tag = f'{length}_{std}'
    return pd.DataFrame({
        f'BBL_{tag}': lower,
        f'BBM_{tag}': mid,
        f'BBU_{tag}': upper,
        f'BBB_{tag}': bwidth,
        f'BBP_{tag}': pctb,
    })


def compute_sma(close: pd.Series, length: int) -> pd.Series:
    return close.rolling(length).mean().rename(f'SMA_{length}')


print('✅ Technical indicator helpers defined (pure NumPy/pandas — no pandas-ta)')

In [ ]:
# ── 7. Stream B — Technical indicators ───────────────────────────────────────
conn  = sqlite3.connect(DB_PATH)
ohlcv = pd.read_sql("""
    SELECT ticker, date, open, high, low, close, volume
    FROM ohlcv ORDER BY ticker, date
""", conn)
conn.close()

ohlcv['date'] = pd.to_datetime(ohlcv['date'])
all_frames = []

for ticker, grp in tqdm(ohlcv.groupby('ticker'), desc='Technical indicators'):
    grp = grp.sort_values('date').copy().reset_index(drop=True)

    # ── Momentum / trend ──────────────────────────────────────────────────────
    grp['RSI_14'] = compute_rsi(grp['close'], length=14)

    macd_df = compute_macd(grp['close'], fast=12, slow=26, signal=9)
    grp = pd.concat([grp, macd_df], axis=1)

    bb_df = compute_bbands(grp['close'], length=20, std=2.0)
    grp = pd.concat([grp, bb_df], axis=1)

    grp['SMA_5']  = compute_sma(grp['close'], 5)
    grp['SMA_10'] = compute_sma(grp['close'], 10)
    grp['SMA_20'] = compute_sma(grp['close'], 20)

    # ── Volume / price structure ──────────────────────────────────────────────
    grp['volume_delta']  = grp['volume'].pct_change()
    grp['overnight_gap'] = (grp['open'] - grp['close'].shift(1)) / grp['close'].shift(1)

    # ── Target label ──────────────────────────────────────────────────────────
    next_close = grp['close'].shift(-1)
    pct        = (next_close - grp['close']) / grp['close']
    grp['price_direction'] = np.where(
        pct >  0.005,  1,
        np.where(pct < -0.005, -1, 0)
    )
    grp.loc[grp.index[-1], 'price_direction'] = np.nan  # no next-day for last row

    # ── Earnings proximity (placeholder = 0; wire in real dates when available) ──
    # See compute_earnings_proximity() helper at the end of the notebook.
    grp['earnings_proximity'] = 0

    grp['ticker'] = ticker
    all_frames.append(grp)

tech_df = pd.concat(all_frames, ignore_index=True)
tech_df.columns = [c.strip() for c in tech_df.columns]
tech_df['date'] = tech_df['date'].dt.strftime('%Y-%m-%d')
print(f'[Stream B] Technical shape: {tech_df.shape}')

In [ ]:
# ── 8. Merge streams ──────────────────────────────────────────────────────────
matrix = tech_df.merge(sentiment_df, on=['ticker', 'date'], how='left')

sent_fill = {
    'mean_sentiment': 0.0, 'sentiment_volatility': 0.0,
    'headline_volume': 0,  'negative_spike_flag': 0,
    'mean_positive': 0.0,  'mean_negative': 0.0, 'mean_neutral': 0.0,
}
matrix = matrix.fillna({k: v for k, v in sent_fill.items() if k in matrix.columns})

FINAL_COLUMNS = [
    'ticker', 'date',
    'mean_sentiment', 'sentiment_volatility', 'headline_volume',
    'negative_spike_flag', 'mean_positive', 'mean_negative', 'mean_neutral',
    'RSI_14',
    'MACD_12_26_9', 'MACDh_12_26_9', 'MACDs_12_26_9',
    'BBL_20_2.0', 'BBM_20_2.0', 'BBU_20_2.0', 'BBB_20_2.0', 'BBP_20_2.0',
    'SMA_5', 'SMA_10', 'SMA_20',
    'close', 'volume', 'volume_delta', 'overnight_gap', 'earnings_proximity',
    'price_direction',
]
available = [c for c in FINAL_COLUMNS if c in matrix.columns]
matrix    = matrix[available]
matrix    = matrix.dropna(subset=['price_direction'])
matrix['price_direction'] = matrix['price_direction'].astype(int)

print(f'Final matrix shape: {matrix.shape}')
print(f'Tickers: {sorted(matrix["ticker"].unique())}')
print(f'Date range: {matrix["date"].min()} → {matrix["date"].max()}')
print(f"Target distribution:\n{matrix['price_direction'].value_counts().to_string()}")
matrix.head()

In [ ]:
# ── 9. Null report ────────────────────────────────────────────────────────────
# NaNs in technical indicators are expected until ~20-26 trading days accumulate
# (warm-up period for slow EMA / Bollinger / SMA_20).
null_report = (matrix.isna().mean() * 100).round(1).rename('null_%')
print(null_report[null_report > 0].to_string() or 'No nulls!')

In [ ]:
# ── 10. Save output ───────────────────────────────────────────────────────────
matrix.to_csv(OUT_PATH, index=False)
print(f'✅ Saved → {OUT_PATH}  ({OUT_PATH.stat().st_size / 1e3:.1f} KB)')

In [ ]:
# ── Helper: earnings proximity (wire in when you have dates) ──────────────────
def compute_earnings_proximity(df: pd.DataFrame, earnings_dates: dict) -> pd.Series:
    """
    earnings_dates = {
        'AAPL': ['2026-05-01', '2026-08-05'],
        'MSFT': ['2026-04-30', '2026-07-29'],
    }
    Usage (inside the per-ticker loop):
        grp['earnings_proximity'] = compute_earnings_proximity(grp, EARNINGS_DATES)
    """
    result = []
    for _, row in df.iterrows():
        dates  = [pd.to_datetime(d) for d in earnings_dates.get(row['ticker'], [])]
        date   = pd.to_datetime(row['date'])
        future = [d for d in dates if d >= date]
        result.append((min(future) - date).days if future else np.nan)
    return pd.Series(result, index=df.index)